In [1]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from protein_data import *

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

model_name = "hugohrban/progen2-medium"
model, tokenizer = initialize_progen2(model_name)

Using cpu device


In [5]:
df_otc_h = pd.read_csv('/Users/johnhutchens/Desktop/Practicum/Data/zInput_Data/DMS_ProteinGym_substitutions/OTC_HUMAN_Lo_2023.csv')

In [6]:
df_otc_h.head()

,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A102E,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.438,1
1,A102G,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.676,1
2,A102P,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.153,0
3,A102S,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.948,1
4,A102T,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.861,1


In [7]:
mut = df_otc_h.iloc[0]['mutant']
seq = df_otc_h.iloc[0]['mutated_sequence']
print(mut)
print(seq)

A102E
MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLKNFTGEEIKYMLWLSADLKFRIKQKGEYLPLLQGKSLGMIFEKRSTRTRLSTETGFELLGGHPCFLTTQDIHLGVNESLTDTARVLSSMADAVLARVYKQSDLDTLAKEASIPIINGLSDLYHPIQILADYLTLQEHYSSLKGLTLSWIGDGNNILHSIMMSAAKFGMHLQAATPKGYEPDASVTKLAEQYAKENGTKLLLTNDPLEAAHGGNVLITDTWISMGQEEEKKKRLQAFQGYQVTMKTAKVAASDWTFLHCLPRKPEEVDDEVFYSPRSLVFPEAENRKWTIMAVMVSLLTDYSPQLQKPKF


In [8]:
len_mut = len(mut)
orig = mut[0]
pos = int(mut[1:len_mut-1])-1
new = mut[len_mut-1]

In [9]:
wild_seq = seq[:pos] + orig + seq[pos+1:]

In [10]:
print(seq[101])
wild_seq[101]

E


'A'

In [11]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/OTC_Human/pg2_otc_hum_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

In [12]:
wild_lp, wild_rlp, wild_llr = collect_log_prob_pg2(wild_seq, model, tokenizer)

In [14]:
wild_lp_saved = pg_dict[None]['log_probs']

In [17]:
print(wild_lp[0])
print(wild_lp_saved[0])

tensor([-3.1000, -4.3898, -3.9654, -3.8103, -3.7488, -3.3572, -4.3612, -3.7033,
        -3.1966, -2.9703, -0.8885, -3.6064, -3.4366, -3.6413, -2.9071, -2.9209,
        -3.3049, -3.5161, -4.9981, -4.5877])
tensor([-3.1000, -4.3898, -3.9654, -3.8103, -3.7488, -3.3572, -4.3612, -3.7033,
        -3.1966, -2.9703, -0.8885, -3.6064, -3.4366, -3.6413, -2.9071, -2.9209,
        -3.3049, -3.5161, -4.9981, -4.5877])


In [20]:
print(wild_lp[150])
print(wild_lp_saved[150])

tensor([-4.7729, -6.7578, -6.0205, -4.5727, -4.5546, -7.5994, -6.6383, -3.1778,
        -5.5534, -0.2210, -3.2509, -6.2598, -6.5063, -4.4152, -5.8187, -5.6404,
        -5.0007, -3.0195, -8.0766, -6.7839])
tensor([-4.7729, -6.7578, -6.0205, -4.5727, -4.5546, -7.5994, -6.6383, -3.1778,
        -5.5534, -0.2210, -3.2509, -6.2598, -6.5063, -4.4152, -5.8187, -5.6404,
        -5.0007, -3.0195, -8.0766, -6.7839])
